
# Lesson 11: Human-in-the-Loop (HITL)

## Learning objectives
- Understand human approval in agentic workflows.
- Use LangGraph's `interrupt()` to pause execution.
- Resume a paused graph using `Command(resume=...)`.
- Use a checkpointer to preserve paused state.
- Route execution based on human approval or rejection.

## Why HITL?

Autonomous agents should not always execute consequential actions without supervision.

Human-in-the-Loop introduces a human decision point into an otherwise automated workflow.

Our example: an agent proposes a ₹5,000 refund and must obtain approval before executing it.

## Workflow
Propose refund → Request approval → Human decision → Execute or reject.


In [1]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

memory = InMemorySaver()

print("HITL imports and checkpointer ready!")

HITL imports and checkpointer ready!



## Defining the refund state

Our graph needs to track:

- `customer`: Customer name.
- `amount`: Requested refund amount.
- `approval`: Human approval decision.
- `status`: Current refund status.

The state is shared between nodes and preserves information as the workflow progresses.


In [2]:
class RefundState(TypedDict):
    customer: str
    amount: float
    approval: str
    status: str

print("Refund state defined!")

Refund state defined!


In [3]:
def propose_refund(state: RefundState):
    print("Preparing refund request...")

    return {
        "status": f"Refund of ₹{state['amount']} proposed for {state['customer']}."
    }


## Understanding interrupt()

`interrupt()` pauses graph execution and exposes a payload to the calling application.

The payload can contain information that a human needs to make a decision.

Execution resumes when the application invokes the graph again using:

`Command(resume=human_decision)`

Important:
- The graph needs a checkpointer.
- The same thread ID must be used when resuming.
- The node containing `interrupt()` starts again when resumed.
- Code before the interrupt may run again, so avoid placing irreversible actions before it.


In [4]:
def request_approval(state: RefundState):
    decision = interrupt({
        "message": "Human approval required",
        "customer": state["customer"],
        "amount": state["amount"],
        "question": "Should this refund be approved?"
    })

    return {
        "approval": decision
    }

In [5]:
def process_refund(state: RefundState):
    if state["approval"] == "approved":
        print("Refund approved!")
        return {"status": "Refund successfully processed."}

    print("Refund rejected!")
    return {"status": "Refund rejected. No money was transferred."}

In [6]:
builder = StateGraph(RefundState)

builder.add_node("propose", propose_refund)
builder.add_node("approval", request_approval)
builder.add_node("process", process_refund)

builder.add_edge(START, "propose")
builder.add_edge("propose", "approval")
builder.add_edge("approval", "process")
builder.add_edge("process", END)

graph = builder.compile(checkpointer=memory)

print("Human approval graph compiled!")

Human approval graph compiled!



## Starting the approval workflow

We will invoke the graph with a customer, refund amount, and initial state.

When execution reaches `interrupt()`, LangGraph pauses the workflow.

The approval node does not finish, and the processing node cannot execute until we resume the graph.


In [8]:
config = {
    "configurable": {
        "thread_id": "refund_101"
    }
}

initial_state = {
    "customer": "Adarsh",
    "amount": 5000.0,
    "approval": "",
    "status": ""
}

result = graph.invoke(initial_state, config=config)

print("Workflow result:")
print(result)

Preparing refund request...
Workflow result:
{'customer': 'Adarsh', 'amount': 5000.0, 'approval': '', 'status': 'Refund of ₹5000.0 proposed for Adarsh.', '__interrupt__': [Interrupt(value={'message': 'Human approval required', 'customer': 'Adarsh', 'amount': 5000.0, 'question': 'Should this refund be approved?'}, id='7fe0bbf539bf64519200c709588b7c26', response_schema=None)]}



## Resuming with human approval

To resume a paused graph, we use:

`Command(resume="approved")`

The same thread ID must be supplied so LangGraph can locate the paused checkpoint.

The graph resumes from the interrupt, receives the human decision, and continues to the processing node.


In [9]:
from langgraph.types import Command

result = graph.invoke(
    Command(resume="approved"),
    config=config
)

print("\nFinal workflow state:")
print(result)

Refund approved!

Final workflow state:
{'customer': 'Adarsh', 'amount': 5000.0, 'approval': 'approved', 'status': 'Refund successfully processed.'}



### Important: Interrupt does not block Python execution

LangGraph's `interrupt()` pauses graph execution, not the Python process or notebook kernel.

The initial `graph.invoke()` returns an `__interrupt__` payload to the calling application.

The application can inspect the request, obtain a human decision, and resume the graph using `Command(resume=decision)`.

The node containing the interrupt resumes from the interrupt point when the graph is continued.

The processing node executes only after the graph resumes.



In [10]:
reject_config = {
    "configurable": {
        "thread_id": "refund_102"
    }
}

rejection_request = {
    "customer": "Rahul",
    "amount": 3000.0,
    "approval": "",
    "status": ""
}

result = graph.invoke(rejection_request, config=reject_config)

print(result)

Preparing refund request...
{'customer': 'Rahul', 'amount': 3000.0, 'approval': '', 'status': 'Refund of ₹3000.0 proposed for Rahul.', '__interrupt__': [Interrupt(value={'message': 'Human approval required', 'customer': 'Rahul', 'amount': 3000.0, 'question': 'Should this refund be approved?'}, id='bbf42f6998dc63b6b99acc6c9286af47', response_schema=None)]}


In [11]:
result = graph.invoke(
    Command(resume="rejected"),
    config=reject_config
)

print("\nFinal state:")
print(result)

Refund rejected!

Final state:
{'customer': 'Rahul', 'amount': 3000.0, 'approval': 'rejected', 'status': 'Refund rejected. No money was transferred.'}


## Approval and Rejection: Key Takeaways

The workflow supports two human decisions:

| Decision | Graph behavior |
|---|---|
| `approved` | Processes the refund |
| `rejected` | Rejects the refund without transferring money |

### Important concepts
- `interrupt()` pauses graph execution and exposes an approval request.
- `Command(resume=...)` resumes the paused graph with the human's decision.
- The same `thread_id` is required to resume the existing workflow.
- A new `thread_id` creates an independent workflow.

### Production safety
Never perform irreversible actions before approval. In a real refund system, verify that the approver is authorized and make refund processing idempotent to prevent duplicate payments.

### Interview question
**Why is a checkpointer required for Human-in-the-Loop workflows?**

It saves the graph's state when execution pauses, allowing the workflow to resume later with the human's decision.